# Convert raw ATD data to CSV

This notebook converts the raw tactical ATD data (the metadata XML and raw tracking text) into a
CSV that can be used by [Frame Seeker](../frame-seeker/) and [data discovery](../data-discovery/).
It writes `data/tactical/BAR-ATH_tactical_ds.csv`.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from kloppy import metrica

import games

paths = games.asset_paths(source='tactical')
dataset = metrica.load_tracking_epts(meta_data=paths['atd_meta'],
                                     raw_data=paths['atd_raw'])
frame_df = dataset.to_df()

csv_path = os.path.join(games.DATA_DIR, 'tactical', 'BAR-ATH_tactical_ds.csv')

ball_speed_by_frame = {}
with open(paths['atd_raw']) as f:
    for line in f:
        frame_id, _players, ball = line.rstrip('\r\n').split(':')
        speed = ball.split(',')[2]
        ball_speed_by_frame[int(frame_id)] = (
            float(speed) if speed != 'NaN' else np.nan
        )

frame_df['ball_speed'] = (
    frame_df['frame_id'].astype(int).map(ball_speed_by_frame)
)

# Add per-player positions in metres, distance to the ball, and team name.
length = dataset.metadata.pitch_dimensions.pitch_length
width = dataset.metadata.pitch_dimensions.pitch_width
ball_x_m = frame_df['ball_x'] * length
ball_y_m = frame_df['ball_y'] * width

extra = {}
for player in dataset.metadata.teams[0].players + dataset.metadata.teams[1].players:
    pid = player.player_id
    x_m = frame_df['%s_x' % pid] * length
    y_m = frame_df['%s_y' % pid] * width
    extra['%s_x_m' % pid] = x_m
    extra['%s_y_m' % pid] = y_m
    extra['%s_ball_dist_m' % pid] = np.hypot(x_m - ball_x_m, y_m - ball_y_m)
    extra['%s_team' % pid] = player.team.name

frame_df = pd.concat([frame_df, pd.DataFrame(extra, index=frame_df.index)], axis=1)
frame_df.to_csv(csv_path, index=False)
print('Wrote %d frames to %s' % (len(frame_df), csv_path))

Wrote 199319 frames to /Users/hafizbadrielubis/Developments/Hubble/more-than-a-hack-2027/data/tactical/BAR-ATH_tactical_ds.csv
